# NeoOLAF Resume From Layer 3, Path-Fixed Version

This notebook resumes a NeoOLAF run after a crash during `layer03_candidate_typing_resolution`.

This version does **not** assume that `/home/...` is accessible from the notebook kernel. It first detects where the notebook is running and tries to find the NeoOLAF root automatically.

Important: the notebook kernel must be able to access the NeoOLAF repository. If the kernel is running on Windows while NeoOLAF is inside Linux/WSL/SSH, switch the Jupyter kernel to the NeoOLAF `.venv` or run Jupyter from inside the NeoOLAF environment.

In [ ]:
from pathlib import Path
import json
import os
import sys
import subprocess
import shlex
import platform
from datetime import datetime

print("Python executable:", sys.executable)
print("Platform:", platform.platform())
print("Current working directory:", Path.cwd())
print("HOME:", os.environ.get("HOME"))
print("USERPROFILE:", os.environ.get("USERPROFILE"))

## 0. Configuration

Usually leave `PROJECT_ROOT = None`. The notebook will try to find the NeoOLAF root from the current directory, common Linux paths, and environment variables.

If auto-detection fails, set `PROJECT_ROOT` manually to the path that exists **inside this notebook kernel**.

In [ ]:
# =========================
# EDIT HERE IF NEEDED
# =========================

# Leave None for auto-detection.
PROJECT_ROOT = None

# Examples, depending on where your Jupyter kernel is running:
# PROJECT_ROOT = Path("/home/galencarmedeiro/git/postdoc/NeoOLAF")  # Linux/WSL/SSH kernel
# PROJECT_ROOT = Path.cwd()                                        # if Jupyter started from NeoOLAF root

# If you know the exact run directory that crashed, put it here.
# Otherwise leave None and the notebook will search recent runs.
RUN_DIR = None

MODEL = "openrouter/openai/gpt-oss-20b"

# Use the same profile you used in the crashed run.
# Examples: "xquality_loose", "xquality_machine32"
PROFILE = "xquality_loose"

FROM_LAYER = 3
TO_LAYER = 12

# Try Layer 3 first. If it fails, set this False and rerun to resume from Layer 2.
PREFER_LAYER03_STATE = True

# Keep same run directory so NeoOLAF can reuse checkpoints.
USE_SAME_RUN_DIR = True
RESUME_SUFFIX = "_resumed_from_layer03"

# Concurrency
MAX_CONCURRENCY_LAYER03 = 4
MAX_CONCURRENCY_LAYER04 = 4
MAX_CONCURRENCY_LAYER05 = 4
MAX_CONCURRENCY_LAYER06 = 4
MAX_CONCURRENCY_LAYER07 = 4
MAX_CONCURRENCY_LAYER08 = 4
MAX_CONCURRENCY_LAYER09 = 4
MAX_CONCURRENCY_LAYER10 = 4
MAX_CONCURRENCY_LAYER11 = 4
MAX_CONCURRENCY_LAYER12 = 4

# Optional RAG backend. Leave None if your crashed command did not use --rag-backend.
RAG_BACKEND = None

VERBOSE = True

## 1. Resolve NeoOLAF project root

This cell detects the repository root. If it fails, the notebook is probably running in the wrong kernel/environment.

In [ ]:
def looks_like_neoolaf_root(path: Path) -> bool:
    if not path.exists() or not path.is_dir():
        return False
    return (
        (path / "experiments").exists()
        or (path / "src" / "neoolaf").exists()
        or (path / "pyproject.toml").exists()
    ) and (
        (path / "examples").exists()
        or (path / "src").exists()
    )

def candidate_project_roots():
    candidates = []

    # Explicit env vars
    for key in ["NEOOLAF_PROJECT_ROOT", "PROJECT_ROOT"]:
        value = os.environ.get(key)
        if value:
            candidates.append(Path(value).expanduser())

    # Current directory and parents
    cwd = Path.cwd()
    candidates.append(cwd)
    candidates.extend(cwd.parents)

    # Common Linux/WSL/SSH paths
    candidates.extend([
        Path("/home/galencarmedeiro/git/postdoc/NeoOLAF"),
        Path.home() / "git" / "postdoc" / "NeoOLAF",
        Path.home() / "NeoOLAF",
    ])

    # Remove duplicates while preserving order
    seen = set()
    unique = []
    for p in candidates:
        try:
            rp = p.resolve()
        except Exception:
            rp = p
        if str(rp) not in seen:
            seen.add(str(rp))
            unique.append(p)
    return unique

if PROJECT_ROOT is None:
    found = []
    for cand in candidate_project_roots():
        if looks_like_neoolaf_root(cand):
            found.append(cand.resolve())
    if found:
        PROJECT_ROOT = found[0]
    else:
        print("Could not auto-detect NeoOLAF root.")
        print("\nTried candidates:")
        for cand in candidate_project_roots():
            print(" -", cand)
        print("\nCurrent Python executable:", sys.executable)
        print("Current working directory:", Path.cwd())
        raise RuntimeError(
            "NeoOLAF root not found from this notebook kernel. "
            "Start Jupyter from /home/galencarmedeiro/git/postdoc/NeoOLAF, "
            "or select the NeoOLAF .venv kernel, "
            "or set PROJECT_ROOT manually to a path visible to this kernel."
        )
else:
    PROJECT_ROOT = Path(PROJECT_ROOT).expanduser().resolve()
    if not looks_like_neoolaf_root(PROJECT_ROOT):
        raise RuntimeError(f"PROJECT_ROOT does not look like NeoOLAF root: {PROJECT_ROOT}")

os.chdir(PROJECT_ROOT)
print("Resolved PROJECT_ROOT:", PROJECT_ROOT)
print("PWD:", Path.cwd())
print("Python executable:", sys.executable)

# Helpful sanity check
if "NeoOLAF" not in str(PROJECT_ROOT):
    print("Warning: PROJECT_ROOT does not contain 'NeoOLAF' in its path. Check it carefully.")

## 2. Find recent NeoOLAF states

This searches under the resolved NeoOLAF root.

In [ ]:
def recent_state_files(root: Path, limit: int = 120):
    states = []
    search_roots = []
    for sub in ["examples", "runs", "."]:
        p = root / sub
        if p.exists():
            search_roots.append(p)

    seen_paths = set()
    for base in search_roots:
        for p in base.rglob("state.json"):
            if p in seen_paths:
                continue
            seen_paths.add(p)
            try:
                stat = p.stat()
            except OSError:
                continue
            states.append((stat.st_mtime, p))
    states.sort(reverse=True)
    return states[:limit]

states = recent_state_files(PROJECT_ROOT, limit=120)

print(f"Found {len(states)} recent state.json files.")
for i, (mtime, p) in enumerate(states[:80], 1):
    try:
        rel = p.relative_to(PROJECT_ROOT)
    except ValueError:
        rel = p
    print(f"{i:02d} | {datetime.fromtimestamp(mtime).strftime('%Y-%m-%d %H:%M:%S')} | {rel}")

## 3. Auto-detect the crashed run directory

It searches for directories containing `layer02_candidate_enrichment` or `layer03_candidate_typing_resolution`.

In [ ]:
LAYER03_NAME = "layer03_candidate_typing_resolution"
LAYER02_NAME = "layer02_candidate_enrichment"

def infer_run_dir_from_state(state_path: Path) -> Path:
    return state_path.parent.parent

def score_state_for_resume(p: Path) -> int:
    s = str(p).lower()
    score = 0
    if LAYER03_NAME in s:
        score += 100
    if LAYER02_NAME in s:
        score += 80
    if "prefix_stop_after_03" in s:
        score += 50
    if "xquality" in s:
        score += 20
    if "layer03" in s:
        score += 10
    return score

if RUN_DIR is None:
    candidates = []
    for mtime, p in states:
        if LAYER03_NAME in str(p) or LAYER02_NAME in str(p):
            candidates.append((score_state_for_resume(p), mtime, infer_run_dir_from_state(p), p))
    candidates.sort(reverse=True)

    if not candidates:
        raise RuntimeError(
            "No Layer 2 or Layer 3 state found. "
            "Check whether your crashed run directory is outside PROJECT_ROOT, "
            "or set RUN_DIR manually."
        )

    print("Top candidate run directories:")
    seen = set()
    displayed = 0
    for rank, (score, mtime, rd, sp) in enumerate(candidates, 1):
        if rd in seen:
            continue
        seen.add(rd)
        displayed += 1
        print(f"{displayed:02d} | score={score} | {datetime.fromtimestamp(mtime).strftime('%Y-%m-%d %H:%M:%S')}")
        print("     RUN_DIR:", rd)
        try:
            print("     state:  ", sp.relative_to(PROJECT_ROOT))
        except ValueError:
            print("     state:  ", sp)
        if displayed >= 20:
            break

    RUN_DIR = candidates[0][2]

RUN_DIR = Path(RUN_DIR).expanduser().resolve()
print("\nSelected RUN_DIR:")
print(RUN_DIR)

if not RUN_DIR.exists():
    raise RuntimeError(f"RUN_DIR does not exist from this kernel: {RUN_DIR}")

## 4. Inspect Layer 2 and Layer 3 states

In [ ]:
def load_json_safe(p: Path):
    try:
        return json.loads(p.read_text(encoding="utf-8"))
    except Exception as e:
        return {"_load_error": repr(e)}

def summarize_state(path: Path):
    if not path.exists():
        print("MISSING:", path)
        return None
    data = load_json_safe(path)
    print("\nSTATE:", path)
    print("mtime:", datetime.fromtimestamp(path.stat().st_mtime).strftime("%Y-%m-%d %H:%M:%S"))
    print("size:", path.stat().st_size, "bytes")
    if isinstance(data, dict):
        for key in ["layer_name", "layer", "status", "completed", "failed_count", "error_count", "run_id"]:
            if key in data:
                print(f"{key}:", data.get(key))
        for key in ["linguistic_expressions", "enriched_expressions", "entity_candidates", "relation_candidates", "candidate_triples"]:
            val = data.get(key)
            if isinstance(val, list):
                print(f"{key}: list[{len(val)}]")
            elif val is not None:
                print(f"{key}: {type(val).__name__}")
    else:
        print("State JSON is not a dict:", type(data))
    return data

LAYER02_STATE = RUN_DIR / LAYER02_NAME / "state.json"
LAYER03_STATE = RUN_DIR / LAYER03_NAME / "state.json"

state2 = summarize_state(LAYER02_STATE)
state3 = summarize_state(LAYER03_STATE)

## 5. Choose resume state

In [ ]:
if PREFER_LAYER03_STATE and LAYER03_STATE.exists():
    RESUME_FROM = LAYER03_STATE
    RESUME_REASON = "Layer 3 state exists, trying to resume from the crashed layer."
elif LAYER02_STATE.exists():
    RESUME_FROM = LAYER02_STATE
    RESUME_REASON = "Using Layer 2 state, rerunning Layer 3 onward."
else:
    raise RuntimeError(
        "No usable Layer 3 or Layer 2 state was found. "
        "Set RUN_DIR manually or check whether checkpoints were disabled."
    )

if USE_SAME_RUN_DIR:
    OUTPUT_RUN_DIR = RUN_DIR
else:
    OUTPUT_RUN_DIR = RUN_DIR.parent / f"{RUN_DIR.name}{RESUME_SUFFIX}"
    OUTPUT_RUN_DIR.mkdir(parents=True, exist_ok=True)

print("RESUME_REASON:", RESUME_REASON)
print("RESUME_FROM:", RESUME_FROM)
print("OUTPUT_RUN_DIR:", OUTPUT_RUN_DIR)

## 6. Build resume command

In [ ]:
cmd = [
    sys.executable, "-m", "neoolaf", "run",
    "--resume-from", str(RESUME_FROM),
    "--run-dir", str(OUTPUT_RUN_DIR),
    "--from-layer", str(FROM_LAYER),
    "--to-layer", str(TO_LAYER),
    "--model", MODEL,
    "--profile", PROFILE,
    "--max-concurrency-layer03", str(MAX_CONCURRENCY_LAYER03),
    "--max-concurrency-layer04", str(MAX_CONCURRENCY_LAYER04),
    "--max-concurrency-layer05", str(MAX_CONCURRENCY_LAYER05),
    "--max-concurrency-layer06", str(MAX_CONCURRENCY_LAYER06),
    "--max-concurrency-layer07", str(MAX_CONCURRENCY_LAYER07),
    "--max-concurrency-layer08", str(MAX_CONCURRENCY_LAYER08),
    "--max-concurrency-layer09", str(MAX_CONCURRENCY_LAYER09),
    "--max-concurrency-layer10", str(MAX_CONCURRENCY_LAYER10),
    "--max-concurrency-layer11", str(MAX_CONCURRENCY_LAYER11),
    "--max-concurrency-layer12", str(MAX_CONCURRENCY_LAYER12),
]

if RAG_BACKEND:
    cmd.extend(["--rag-backend", RAG_BACKEND])

if VERBOSE:
    cmd.append("--verbose")

print("Command:")
print(" ".join(shlex.quote(x) for x in cmd))

## 7. Run resume

If resuming from Layer 3 fails, set `PREFER_LAYER03_STATE = False` in the configuration cell and rerun. That resumes from Layer 2 and reruns Layer 3 onward.

In [ ]:
env = os.environ.copy()

print("Starting resume at", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
result = subprocess.run(cmd, cwd=PROJECT_ROOT, env=env, check=False)
print("Finished with return code:", result.returncode)

if result.returncode != 0:
    print("\nThe command failed.")
    print("If it failed when resuming from Layer 3 state, set PREFER_LAYER03_STATE=False and rerun from Layer 2.")
    raise SystemExit(result.returncode)

## 8. Check final states and Layer 12 exports

In [ ]:
states_after = recent_state_files(OUTPUT_RUN_DIR, limit=80)
print(f"Latest states under {OUTPUT_RUN_DIR}:")
for i, (mtime, p) in enumerate(states_after[:50], 1):
    try:
        rel = p.relative_to(PROJECT_ROOT)
    except ValueError:
        rel = p
    print(f"{i:02d} | {datetime.fromtimestamp(mtime).strftime('%Y-%m-%d %H:%M:%S')} | {rel}")

print("\nPossible Layer 12/export files:")
for p in OUTPUT_RUN_DIR.rglob("*"):
    if p.is_file() and p.name in {
        "kg_local.json",
        "kg_inferred.json",
        "ontology_local.ttl",
        "ontology_inferred.ttl",
        "state.json",
    }:
        s = str(p).lower()
        if "layer12" in s or "exports" in s:
            print(p)